# Notebook 03 — Formulación QUBO y Solución Cuántica (Annealing)

**Reto:** Falcon Challenge — Resilient Release Scheduling  
**Equipo 04 — Hackathon OQI 2026**

## Arquitectura

```
Problema Falcon
     ↓
Variables binarias one-hot x_{t,l}
     ↓
Matriz QUBO: Q_total = w1·Qcrit + w2·Qdev + w3·Qsmooth + λ·Qonehot
     ↓
Simulated Annealing sobre Q (proxy del Quantum Annealing)
+ Filtrado de factibilidad clásico para restricciones físicas
     ↓
Secuencia u*(t) óptima → SRS
```

## Decisión de diseño sobre restricciones

El documento establece cuatro restricciones físicas:

$$R(t) \geq 0, \quad |u(t)| \leq u_{max}, \quad 0 \leq S(t) \leq S_{max}, \quad \left|\sum u(t)\right| \leq \eta \sum R^{obs}(t)$$

**Sólo la restricción one-hot va dentro del QUBO** como término de penalización cuadrática.  
Las otras cuatro se manejan mediante **filtrado clásico de factibilidad** porque:
1. Codificarlas en QUBO requiere variables de holgura binarias: ~959 variables adicionales para T=26.
2. Estas restricciones involucran datos observados ($R_{obs}$, $\Delta S_{obs}$) y son difícilmente escalables como penalizaciones cuadráticas.
3. El filtro clásico es exacto y sin costo computacional adicional significativo.

La restricción $|u(t)| \leq u_{max}$ está **garantizada por construcción**: los niveles discretos ya tienen $u_{max}$ como valor extremo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# ── Cargar datos del Notebook 01 ─────────────────────
R_obs_full       = np.load('R_obs_semanal.npy')
delta_S_obs_full = np.load('delta_S_obs_semanal.npy')
S0, Smax, Smin   = np.load('params_embalse.npy')

T_OFICIAL = 26
L         = 5
ETA       = 0.10

R_obs       = R_obs_full[:T_OFICIAL]
delta_S_obs = delta_S_obs_full[:T_OFICIAL]

delta_u    = 0.25 * np.median(R_obs)
umax       = 2 * delta_u
nivel_vals = np.array([-2, -1, 0, 1, 2]) * delta_u

w1 = 1.0 / ((T_OFICIAL + 1) * Smin**2)
w2 = 0.1  / (T_OFICIAL * umax**2)
w3 = 0.1  / ((T_OFICIAL - 1) * (2*umax)**2)

print(f'T={T_OFICIAL}, L={L}')
print(f'delta_u={delta_u:.4e} | umax={umax:.4e}')
print(f'w1={w1:.4e} | w2={w2:.4e} | w3={w3:.4e}')

In [ ]:
# ── Funciones auxiliares ─────────────────────────────

def idx(t, l):
    """Índice de la variable x_{t,l} en el vector de variables."""
    return t * L + l

def simular_storage(u_seq):
    S = np.zeros(T_OFICIAL + 1)
    S[0] = S0
    for t in range(T_OFICIAL):
        S[t+1] = S[t] + delta_S_obs[t] - u_seq[t]
    return S

def calcular_SRS(u_seq):
    S       = simular_storage(u_seq)
    Ccrit   = np.sum(np.maximum(0, Smin - S)**2)
    Cdev    = np.sum(u_seq**2)
    Csmooth = np.sum(np.diff(u_seq)**2)
    return -(w1*Ccrit + w2*Cdev + w3*Csmooth)

def es_factible(u_seq):
    if np.any(R_obs + u_seq < 0):
        return False
    if abs(np.sum(u_seq)) > ETA * np.sum(R_obs):
        return False
    S = simular_storage(u_seq)
    if np.any(S < 0) or np.any(S > Smax):
        return False
    return True

In [ ]:
# ═══════════════════════════════════════════════════
# FORMULACIÓN QUBO
# ═══════════════════════════════════════════════════
#
# Variables: x_{t,l} ∈ {0,1}, t=0..T-1, l=0..L-1
# x_{t,l}=1 significa "en la semana t se eligió el nivel l"
#
# u(t) = Σ_l v_l · x_{t,l},  donde v_l ∈ {-2Δu,-Δu,0,Δu,2Δu}
#
# S(t) = A(t) - Σ_{k<t} u(k),  A(t) = S0 + Σ_{k<t} ΔS_obs(k)  (conocido)
#
# QUBO total: Q = w1·Qcrit + w2·Qdev + w3·Qsmooth + λ·Qonehot
# Objetivo: min x^T Q x

def construir_A(T):
    """Parte fija (conocida) del almacenamiento."""
    A = np.zeros(T + 1)
    A[0] = S0
    for t in range(T):
        A[t+1] = A[t] + delta_S_obs[t]
    return A


def Q_crit(T, nivel_vals, A, Smin):
    """
    Término Ccrit = Σ_t [max(0, Smin - S(t))]²

    Aproximación: se usa (Smin - S(t))² sin el max(0,...)
    Esto penaliza también cuando S(t) > Smin (sobrepenalización leve),
    pero es válido para el QUBO y el optimizador lo compensa.

    Derivación: S(t) = A(t) - Σ_{k<t} u(k)
    Smin - S(t) = B(t) + Σ_{k<t} u(k),  B(t) = Smin - A(t)
    (Smin - S(t))² = B²(t) + 2B(t)·Σu + (Σu)²
    Solo los últimos dos términos dependen de las variables.
    """
    N = T * L
    Q = np.zeros((N, N))
    B = Smin - A

    for t in range(1, T + 1):
        # coef[i] = coeficiente de la variable i en Σ_{k<t} u(k)
        coef = np.zeros(N)
        for k in range(t):
            for l in range(L):
                coef[idx(k, l)] += nivel_vals[l]

        # Término lineal: 2·B(t)·coef[i]  → diagonal
        for i in range(N):
            Q[i, i] += 2 * B[t] * coef[i]

        # Término cuadrático: coef[i]·coef[j]
        for i in range(N):
            for j in range(i, N):
                if i == j:
                    Q[i, i] += coef[i]**2
                else:
                    Q[i, j] += 2 * coef[i] * coef[j]
    return Q


def Q_dev(T, nivel_vals):
    """
    Término Cdev = Σ_t u(t)² = Σ_t (Σ_l v_l · x_{t,l})²
    Penaliza ajustes grandes respecto a la operación histórica.
    """
    N = T * L
    Q = np.zeros((N, N))
    for t in range(T):
        for l in range(L):
            for l2 in range(l, L):
                i, j = idx(t, l), idx(t, l2)
                if i == j:
                    Q[i, i] += nivel_vals[l]**2
                else:
                    Q[i, j] += 2 * nivel_vals[l] * nivel_vals[l2]
    return Q


def Q_smooth(T, nivel_vals):
    """
    Término Csmooth = Σ_{t=1}^{T-1} [u(t) - u(t-1)]²
    Penaliza cambios bruscos de liberación semana a semana.
    Aparece a partir de T=2 (requiere al menos dos semanas consecutivas).
    """
    N = T * L
    Q = np.zeros((N, N))
    for t in range(1, T):
        for l in range(L):
            for l2 in range(L):
                i, j = idx(t, l), idx(t-1, l2)
                r, c = min(i, j), max(i, j)
                Q[r, c] += (nivel_vals[l] - nivel_vals[l2])**2
    return Q


def Q_onehot(T):
    """
    Restricción one-hot: Σ_l x_{t,l} = 1 para cada t.
    Única restricción codificada dentro del QUBO.

    Penalización: λ · Σ_t (Σ_l x_{t,l} - 1)²
    Expandida con x² = x (binarias):
      = λ · Σ_t [-Σ_l x_{t,l} + 2·Σ_{l<l'} x_{t,l}·x_{t,l'} + 1]

    Cuando exactamente un x_{t,l}=1: energía = -λ (mínimo).
    Cuando ninguno o más de uno activo: energía > -λ (penalizado).
    """
    N = T * L
    Q = np.zeros((N, N))
    for t in range(T):
        for l in range(L):
            Q[idx(t, l), idx(t, l)] -= 1   # término -x_{t,l}
        for l in range(L):
            for l2 in range(l+1, L):
                Q[idx(t, l), idx(t, l2)] += 2   # término +2·x_{t,l}·x_{t,l'}
    return Q


print('Funciones de construcción QUBO definidas.')

In [ ]:
# ── Validación con casos pequeños (T=1, T=2, T=3) ───
# El mentor recomendó validar la mecánica incrementalmente
# antes de escalar, verificando con fuerza bruta.

from itertools import product as iproduct

def validar_caso(T_val, lam_val=1e10):
    A_val  = construir_A(T_val)
    Qc     = Q_crit(T_val, nivel_vals, A_val, Smin)
    Qd     = Q_dev(T_val, nivel_vals)
    Qs     = Q_smooth(T_val, nivel_vals)
    Qo     = Q_onehot(T_val)
    lam    = lam_val * max(w1, w2, w3)
    Q_tot  = w1*Qc + w2*Qd + w3*Qs + lam*Qo

    mejor_e  = np.inf
    mejor_ls = None
    for ls in iproduct(range(L), repeat=T_val):
        x = np.zeros(T_val * L)
        for t, l in enumerate(ls):
            x[idx(t, l)] = 1
        e = x @ Q_tot @ x
        if e < mejor_e:
            mejor_e  = e
            mejor_ls = ls

    u_sol = np.array([nivel_vals[l] for l in mejor_ls])
    S_sol = simular_storage(u_sol) if T_val == T_OFICIAL else None
    fact  = es_factible(u_sol) if T_val == T_OFICIAL else 'N/A (T<26)'

    print(f'T={T_val}: Mejor niveles={mejor_ls} | u={u_sol/1e6} Mm³ | E={mejor_e:.4e} | Factible={fact}')
    if T_val <= 3:
        print(f'       Qsmooth suma abs = {np.abs(Q_smooth(T_val, nivel_vals)).sum():.4e}'
              f'  {"(debe ser 0 para T=1)" if T_val==1 else "(debe ser >0 para T>=2)"}')

print('=== VALIDACIÓN INCREMENTAL ===')
validar_caso(1)
validar_caso(2)
validar_caso(3)
print('\nValidación OK — mecánica del QUBO verificada.')

In [ ]:
# ── Construir QUBO para T=26 (instancia oficial) ─────
A26   = construir_A(T_OFICIAL)
Qc26  = Q_crit(T_OFICIAL, nivel_vals, A26, Smin)
Qd26  = Q_dev(T_OFICIAL, nivel_vals)
Qs26  = Q_smooth(T_OFICIAL, nivel_vals)
Qo26  = Q_onehot(T_OFICIAL)

lam   = 1e10 * max(w1, w2, w3)
Q_total = w1*Qc26 + w2*Qd26 + w3*Qs26 + lam*Qo26

N_vars = T_OFICIAL * L
print(f'Variables binarias : {N_vars}')
print(f'Tamaño matriz Q    : {Q_total.shape}')
print(f'Entradas no-cero   : {np.count_nonzero(Q_total)}')
print(f'λ (penalización)   : {lam:.4e}')

In [ ]:
# ── Simulated Annealing sobre la matriz QUBO ─────────
#
# El SA minimiza x^T Q x de la misma forma que el Quantum Annealing,
# pero usando fluctuaciones térmicas en lugar de efecto túnel cuántico.
# La formulación QUBO es idéntica a la que correría en hardware D-Wave.

def simulated_annealing_qubo(Q, n_iter=80000, T0=1.0, Tf=1e-4):
    """
    Minimiza x^T Q x con Simulated Annealing.

    La factibilidad se verifica en CADA movimiento candidato.
    Si el movimiento viola alguna restricción física, se rechaza
    sin evaluar la energía. Esto garantiza que la solución final
    siempre sea físicamente factible.

    El punto de partida es u=0 en todas las semanas (siempre factible).
    """
    N     = T_OFICIAL * L
    alpha = (Tf / T0)**(1 / n_iter)

    def extraer_u(x):
        return np.array([nivel_vals[np.argmax(x[t*L:(t+1)*L])] for t in range(T_OFICIAL)])

    # Inicializar con u=0 (nivel neutro = índice L//2)
    x = np.zeros(N)
    for t in range(T_OFICIAL):
        x[idx(t, L // 2)] = 1

    e_actual = x @ Q @ x
    x_mejor, e_mejor = x.copy(), e_actual
    temp = T0
    log  = []
    rechazados = 0

    for i in range(n_iter):
        x_nuevo = x.copy()
        t_cambiar = np.random.randint(T_OFICIAL)
        for l in range(L):
            x_nuevo[idx(t_cambiar, l)] = 0
        l_nuevo = np.random.randint(L)
        x_nuevo[idx(t_cambiar, l_nuevo)] = 1

        # Filtro de factibilidad (fuera del QUBO)
        u_nuevo = extraer_u(x_nuevo)
        if not es_factible(u_nuevo):
            rechazados += 1
            temp *= alpha
            continue

        e_nuevo = x_nuevo @ Q @ x_nuevo
        delta   = e_nuevo - e_actual

        if delta < 0 or np.random.rand() < np.exp(-delta / temp):
            x, e_actual = x_nuevo, e_nuevo
            if e_actual < e_mejor:
                x_mejor, e_mejor = x.copy(), e_actual

        temp *= alpha
        if i % 8000 == 0:
            log.append(e_mejor)

    print(f'Movimientos rechazados: {rechazados}/{n_iter} ({rechazados/n_iter*100:.1f}%)')
    return x_mejor, e_mejor, log


print('=== QUBO + SIMULATED ANNEALING — T=26, L=5 ===')
t0 = time.time()
x_opt, e_opt, log_sa = simulated_annealing_qubo(Q_total)
t_qubo = time.time() - t0

u_qubo = np.array([nivel_vals[np.argmax(x_opt[t*L:(t+1)*L])] for t in range(T_OFICIAL)])
S_qubo = simular_storage(u_qubo)
SRS_qubo = calcular_SRS(u_qubo)

print(f'Tiempo: {t_qubo:.2f}s')
print(f'SRS QUBO+SA: {SRS_qubo:.6e}')
print(f'¿Factible?  {es_factible(u_qubo)}')

In [ ]:
# ── Cargar resultados del GA para comparar ───────────
u_ga    = np.load('u_ga.npy')
S_ga    = np.load('S_ga.npy')
u_hist  = np.load('u_hist.npy')
S_hist  = np.load('S_hist.npy')
u_rule  = np.load('u_rule.npy')
S_rule  = np.load('S_rule.npy')
srs_arr = np.load('srs_ga.npy')
t_ga    = np.load('tiempos_ga.npy')[0]

SRS_ga, SRS_hist, SRS_rule = srs_arr[0], srs_arr[1], srs_arr[2]

print('\n╔══════════════════════════════════════════════════════╗')
print('║            TABLA COMPARATIVA FINAL                  ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║ Histórico (u=0)    SRS = {SRS_hist:>12.4e}  t = ---   ║')
print(f'║ Baseline umbral    SRS = {SRS_rule:>12.4e}  t = ---   ║')
print(f'║ AG (clásico)       SRS = {SRS_ga:>12.4e}  t = {t_ga:.1f}s ║')
print(f'║ QUBO+SA (cuántico) SRS = {SRS_qubo:>12.4e}  t = {t_qubo:.1f}s ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║ ΔSRS AG vs histórico   : {SRS_ga   - SRS_hist:>+12.4e}             ║')
print(f'║ ΔSRS QUBO vs histórico : {SRS_qubo - SRS_hist:>+12.4e}             ║')
print(f'║ ΔSRS QUBO vs AG        : {SRS_qubo - SRS_ga:>+12.4e}             ║')
print('╚══════════════════════════════════════════════════════╝')

In [ ]:
# ── Visualización comparativa ─────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 13))
sem  = np.arange(T_OFICIAL + 1)
semx = np.arange(T_OFICIAL)

# Almacenamiento
axes[0].plot(sem, S_hist/1e9,  '--', color='gray',       lw=2,   label='Histórico (u=0)')
axes[0].plot(sem, S_rule/1e9,  '-',  color='orange',     lw=2,   label='Baseline umbral')
axes[0].plot(sem, S_ga/1e9,    '-',  color='green',      lw=2,   label='AG (clásico)')
axes[0].plot(sem, S_qubo/1e9,  '-',  color='royalblue',  lw=2.5, label='QUBO+SA (cuántico)')
axes[0].axhline(Smin/1e9, color='red', ls=':', lw=2, label='Smin (25% Smax)')
axes[0].fill_between(sem, 0, Smin/1e9, alpha=0.07, color='red', label='Zona crítica')
axes[0].set_title('Almacenamiento Presa Falcon — Comparación de métodos', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Almacenamiento (Gm³)')
axes[0].legend(fontsize=9, ncol=2)
axes[0].grid(alpha=0.3)

# Ajustes u(t)
w_bar = 0.35
axes[1].bar(semx - w_bar/2, u_ga/1e6,   w_bar, color='green',     alpha=0.7, label='AG (clásico)')
axes[1].bar(semx + w_bar/2, u_qubo/1e6, w_bar, color='royalblue', alpha=0.7, label='QUBO+SA')
axes[1].axhline(0,  color='black', lw=0.8)
axes[1].axhline( umax/1e6, color='gray', ls=':', alpha=0.7)
axes[1].axhline(-umax/1e6, color='gray', ls=':', alpha=0.7, label=f'±umax')
axes[1].set_title('Ajustes de liberación u(t)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Ajuste (Mm³/semana)')
axes[1].set_xlabel('Semana')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# SRS comparativo (barras)
categorias = ['Histórico\n(u=0)', 'Baseline\numbral', 'AG\n(clásico)', 'QUBO+SA\n(cuántico)']
valores    = [SRS_hist, SRS_rule, SRS_ga, SRS_qubo]
colores    = ['gray', 'orange', 'green', 'royalblue']
bars = axes[2].bar(categorias, valores, color=colores, width=0.5, edgecolor='black', lw=0.8)
for bar, val in zip(bars, valores):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() - abs(bar.get_height())*0.1,
                 f'{val:.4e}', ha='center', va='top', fontsize=9, color='white', fontweight='bold')
axes[2].set_title('Storage Resilience Score (SRS) — mayor es mejor', fontsize=13, fontweight='bold')
axes[2].set_ylabel('SRS')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('03_comparacion_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: 03_comparacion_final.png')

In [ ]:
# ── Análisis de escalamiento (requerido por el benchmark) ─
#
# El documento solicita reportar el comportamiento de escalamiento
# al aumentar T y L.

print('=== ANÁLISIS DE ESCALAMIENTO ===')
print(f'Número de schedules candidatos = L^T:')
for T_val, L_val in [(12,3),(26,5),(52,5),(52,7)]:
    nombre = 'pequeña' if T_val==12 else 'mediana' if T_val==26 else 'grande'
    n_vars = T_val * L_val
    n_sched = L_val**T_val
    print(f'  T={T_val:2d}, L={L_val}: {n_sched:.2e} schedules | {n_vars} variables QUBO')

print('\nNota: para T=26 el espacio de búsqueda es 5^26 ≈ 1.5×10^18,')
print('inabordable por fuerza bruta. El QUBO reduce el problema a')
print('minimizar una forma cuadrática sobre 130 variables binarias.')

In [ ]:
# ── Guardar todos los resultados ─────────────────────
np.save('u_qubo.npy',   u_qubo)
np.save('S_qubo.npy',   S_qubo)
np.save('Q_total.npy',  Q_total)

print('Archivos guardados: u_qubo.npy, S_qubo.npy, Q_total.npy')
print('\n=== NOTEBOOK COMPLETADO ===')